# Unidad 3 – Sesión 3
## Selección de variables y creación de nuevas características

**Curso:** Minería de Datos  
**Fase CRISP-DM:** Preparación de los Datos

### Propósito
- Decidir qué variables conservar, excluir o transformar;
- Identificar variables identificadoras, irrelevantes o redundantes;
- Crear nuevas variables útiles;
- Comprender feature engineering;
- Construir un dataset analítico final.

> **Idea clave:** No todas las variables disponibles deben llegar al dataset final.

In [ ]:
import pandas as pd
import numpy as np

data = {
    "cliente_id": [1, 2, 3, 4, 5],
    "edad": [25, 32, 40, 29, 36],
    "ingreso_mensual": [1500, 3200, 4200, 2100, 3800],
    "gasto_mensual": [900, 1800, 2600, 1200, 2500],
    "fecha_registro": ["2025-01-10","2025-05-15","2024-08-01","2026-01-20","2024-12-05"],
    "ciudad": ["LIMA", "CUSCO", "LIMA", "AREQUIPA", "CUSCO"],
    "telefono": ["999111222","988222333","977333444","966444555","955555666"]
}

df = pd.DataFrame(data)
df

# 1. Selección de variables

Pregúntate:

> ¿Esta variable aporta información relevante para el objetivo del proyecto?

Las variables pueden:
- Conservarse;
- Excluirse;
- Transformarse;
- Utilizarse para crear nuevas variables.

# 2. Identificadores

In [ ]:
df[["cliente_id", "telefono"]]

`cliente_id` y `telefono` identifican registros, pero normalmente no aportan valor predictivo directo.

No tiene sentido interpretar `cliente 5 > cliente 1`.

# 3. Variables redundantes

In [ ]:
df["ingreso_anual"] = df["ingreso_mensual"] * 12
df[["ingreso_mensual","ingreso_anual"]]

Dos variables pueden contener prácticamente la misma información. Eso puede indicar redundancia.

# 4. Correlación entre variables numéricas

In [ ]:
df[["edad","ingreso_mensual","gasto_mensual","ingreso_anual"]].corr()

Una correlación alta puede sugerir redundancia, pero no obliga automáticamente a eliminar una variable. También importa el significado de negocio.

# 5. Feature Engineering

**Feature Engineering** consiste en crear nuevas variables a partir de información existente para representar mejor el fenómeno analizado.

# 6. Variable numérica derivada

In [ ]:
df["porcentaje_gasto"] = (
    df["gasto_mensual"] / df["ingreso_mensual"]
) * 100

df[["ingreso_mensual","gasto_mensual","porcentaje_gasto"]]

`porcentaje_gasto` responde: ¿qué proporción del ingreso mensual utiliza el cliente en gastos?

# 7. Transformación temporal

In [ ]:
df["fecha_registro"] = pd.to_datetime(df["fecha_registro"], errors="coerce")
df["anio_registro"] = df["fecha_registro"].dt.year
df["mes_registro"] = df["fecha_registro"].dt.month

df[["fecha_registro","anio_registro","mes_registro"]]

# 8. Crear antigüedad

In [ ]:
fecha_referencia = pd.Timestamp("2026-11-01")

df["antiguedad_dias"] = (
    fecha_referencia - df["fecha_registro"]
).dt.days

df[["fecha_registro","antiguedad_dias"]]

# 9. Discretización o Binning

In [ ]:
df["grupo_edad"] = pd.cut(
    df["edad"],
    bins=[0,25,35,50,100],
    labels=["18-25","26-35","36-50","51+"],
    include_lowest=True
)

df[["edad","grupo_edad"]]

La discretización transforma una variable numérica en grupos. Mejora interpretación, pero puede perder detalle.

In [ ]:
df.groupby("grupo_edad", observed=False)["gasto_mensual"].agg(
    cantidad="count",
    promedio="mean",
    mediana="median"
)

# 10. Variable categórica derivada

In [ ]:
df["nivel_ingreso"] = pd.cut(
    df["ingreso_mensual"],
    bins=[0,2000,3500,float("inf")],
    labels=["BAJO","MEDIO","ALTO"]
)

df[["ingreso_mensual","nivel_ingreso"]]

# 11. Variables agregadas

In [ ]:
ventas = pd.DataFrame({
    "cliente_id": [1,1,1,2,2,3,3,3,3],
    "factura_id": ["F1","F2","F3","F4","F5","F6","F7","F8","F9"],
    "monto": [100,150,200,80,120,50,70,90,110]
})

ventas

In [ ]:
resumen_cliente = ventas.groupby("cliente_id").agg(
    total_compras=("monto","sum"),
    promedio_compra=("monto","mean"),
    cantidad_compras=("factura_id","nunique")
).reset_index()

resumen_cliente

Aquí cambia la granularidad:

**Antes:** una fila = una transacción  
**Después:** una fila = un cliente

# 12. Variables constantes

In [ ]:
df["pais"] = "PERU"
df["pais"].value_counts()

Una variable constante no distingue observaciones y suele aportar poca información analítica.

# 13. Variables con poca variabilidad

In [ ]:
ejemplo_binario = pd.Series(["SI"] * 98 + ["NO"] * 2)
ejemplo_binario.value_counts(normalize=True) * 100

Poca variabilidad no significa eliminar automáticamente la variable. Puede representar una clase rara importante.

# 14. Matriz de selección

In [ ]:
matriz_seleccion = pd.DataFrame({
    "Variable": [
        "cliente_id","edad","ingreso_mensual",
        "telefono","fecha_registro","porcentaje_gasto"
    ],
    "Rol": [
        "Identificador","Numérica","Numérica",
        "Identificador","Temporal","Derivada numérica"
    ],
    "Calidad": ["Alta"] * 6,
    "Relevancia": [
        "Baja","Alta","Alta","Baja","Alta","Alta"
    ],
    "Decision": [
        "Excluir como predictor","Conservar","Conservar",
        "Excluir","Transformar","Conservar"
    ],
    "Justificacion": [
        "Solo identifica registros",
        "Puede aportar información del cliente",
        "Representa capacidad económica",
        "No aporta valor analítico directo",
        "Es más útil transformarla a antigüedad",
        "Resume la relación ingreso/gasto"
    ]
})

matriz_seleccion

# 15. Dataset analítico final

In [ ]:
variables_finales = [
    "edad",
    "ingreso_mensual",
    "gasto_mensual",
    "porcentaje_gasto",
    "antiguedad_dias",
    "grupo_edad",
    "nivel_ingreso",
    "ciudad"
]

df_analitico = df[variables_finales].copy()
df_analitico

# 16. Validación

In [ ]:
df_analitico.dtypes

In [ ]:
df_analitico.isnull().sum()

In [ ]:
df_analitico.describe(include="all")

Verifica:
- Tipos
- Nulos
- Sentido de las nuevas variables
- Categorías
- Rangos
- Coherencia con variables originales

# 17. Actividad práctica con dataset propio

In [ ]:
import pandas as pd

df_proyecto = pd.read_csv(
    "/srv/jupyter-data/Compartido/Minería/LI-Small_Trans.csv",
    dtype={"From Bank": str, "To Bank": str, "Account": str, "Account.1": str},
)
print(df_proyecto.shape)
df_proyecto.head()

In [ ]:
# Transformaciones y variables nuevas sobre el dataset real
df_proyecto["Timestamp"] = pd.to_datetime(
    df_proyecto["Timestamp"], format="%Y/%m/%d %H:%M", errors="coerce"
)

# Numérica derivada: log de monto (distribución muy asimétrica)
df_proyecto["monto_pagado_log"] = np.log1p(df_proyecto["Amount Paid"])

# Temporales
df_proyecto["hora"] = df_proyecto["Timestamp"].dt.hour
df_proyecto["dia_semana"] = df_proyecto["Timestamp"].dt.day_name()

# Categórica derivada: rango de monto (la relación con el lavado no es lineal)
df_proyecto["rango_monto"] = pd.cut(
    df_proyecto["Amount Paid"],
    bins=[0, 1e3, 1e4, 1e5, float("inf")],
    labels=["BAJO", "MEDIO", "ALTO", "MUY_ALTO"],
)

# Binaria: cambio de moneda
df_proyecto["misma_moneda"] = (
    df_proyecto["Receiving Currency"] == df_proyecto["Payment Currency"]
).astype(int)

nuevas = ["monto_pagado_log", "hora", "dia_semana", "rango_monto", "misma_moneda"]
print(df_proyecto[nuevas].isnull().sum())
df_proyecto[["Timestamp"] + nuevas].head()

Cada grupo debe identificar:
- 2 variables que conservará;
- 2 variables que excluirá;
- 2 variables que transformará;
- mínimo 2 variables nuevas que creará.

In [ ]:
matriz_proyecto = pd.DataFrame({
    "Variable": [
        "Timestamp", "From Bank / To Bank", "Account / Account.1",
        "Amount Paid", "Amount Received",
        "Receiving Currency / Payment Currency", "Payment Format", "Is Laundering",
    ],
    "Rol": [
        "Temporal", "Identificador", "Identificador",
        "Numérica", "Numérica", "Categórica", "Categórica", "Objetivo",
    ],
    "Calidad": ["Alta"] * 8,
    "Relevancia": ["Alta", "Baja", "Baja", "Alta", "Baja", "Media", "Alta", "Alta"],
    "Decision": [
        "Transformar", "Excluir", "Excluir",
        "Transformar", "Excluir", "Transformar", "Conservar", "Conservar (objetivo)",
    ],
    "Justificacion": [
        "Se extraen hora y día de semana",
        "Código de banco: identifica, no cuantifica",
        "Identifica cuenta; no aporta patrón general",
        "Se logaritmiza (monto_pagado_log) y se categoriza (rango_monto)",
        "Igual a Amount Paid en 98.6 % de filas: redundante",
        "Se resume en misma_moneda",
        "Señal fuerte: ACH concentra el lavado",
        "Variable a predecir",
    ],
})

matriz_proyecto

# 18. Feature Engineering obligatorio

Variables creadas sobre el dataset de transacciones:

| Tipo | Variable | Origen | Cómo se creó |
|---|---|---|---|
| Numérica derivada | `monto_pagado_log` | `Amount Paid` | `np.log1p` (reduce la asimetría de los montos) |
| Temporal | `hora`, `dia_semana` | `Timestamp` | `.dt.hour` y `.dt.day_name()` |
| Categórica derivada | `rango_monto` | `Amount Paid` | `pd.cut` en BAJO, MEDIO, ALTO y MUY_ALTO |
| Binaria | `misma_moneda` | `Receiving Currency`, `Payment Currency` | comparación `==` |

# 19. Matriz final

In [ ]:
print(df_proyecto["Is Laundering"].value_counts())
print(df_proyecto.shape)

# Dataset analítico final del proyecto
variables_proyecto = ["hora", "dia_semana", "rango_monto", "misma_moneda", "Payment Format", "Is Laundering"]
df_analitico_proyecto = df_proyecto[variables_proyecto].copy()
print(df_analitico_proyecto.dtypes)
print(df_analitico_proyecto.isnull().sum())
df_analitico_proyecto.head()

In [ ]:
matriz_final = pd.DataFrame({
    "Variable_original": [
        "Timestamp", "Amount Paid", "Receiving Currency / Payment Currency",
        "Payment Format", "From Bank / To Bank", "Account / Account.1",
        "Amount Received", "Is Laundering",
    ],
    "Decision": [
        "Transformar", "Transformar", "Transformar",
        "Conservar", "Excluir", "Excluir", "Excluir", "Objetivo",
    ],
    "Variables_resultantes": [
        "hora, dia_semana", "monto_pagado_log, rango_monto", "misma_moneda",
        "Payment Format (One-Hot)", "—", "—", "—", "Is Laundering",
    ],
    "Motivo": [
        "Patrón horario y semanal de transacciones",
        "Log reduce asimetría; rango captura relación no lineal",
        "Cambio de moneda: sin lavado",
        "ACH concentra el lavado",
        "Identificador de banco",
        "Identificador de cuenta",
        "Redundante con Amount Paid",
        "Variable a predecir",
    ],
})

matriz_final

# 20. Producto final

Cada grupo debe entregar:
- matriz de selección;
- variables conservadas;
- variables excluidas;
- variables transformadas;
- mínimo 2 variables nuevas;
- dataset analítico final;
- conclusión de máximo 5 líneas.

> ¿Qué variables fueron seleccionadas o creadas y por qué mejoran la utilidad del dataset?
> Las transformaciones fueron necesarias porque el dataset tiene identificadores (bancos, cuentas) y montos muy asimétricos que no se pueden usar tal cual. Excluimos 3 variables redundantes o sin valor predictivo, transformamos 3 (fecha, monto y monedas) en 5 variables nuevas y conservamos el método de pago, que es la señal más fuerte del lavado (ACH). El dataset final (6,924,049 filas, 16 columnas) no tiene nulos, pero el lavado es solo el 0.05 %, así que cualquier modelo deberá manejar ese desbalance. Estas variables mejoran la utilidad del dataset porque separan el patrón temporal, el tamaño del monto y el tipo de pago que sí se asocian con transacciones sospechosas.

# 22. Cierre

### Sesión 1: Limpiar
- nulos
- duplicados
- inconsistencias
- inválidos
- outliers

### Sesión 2: Transformar
- tipos
- fechas
- categorías
- escalamiento

### Sesión 3: Seleccionar y crear
- selección de variables
- redundancia
- feature engineering
- discretización
- agregaciones
- dataset analítico final

> **Preparar datos significa construir un conjunto de variables relevante, confiable y adecuado para el objetivo del proyecto.**